# From High Quality Data Models to High Fidelity Network Models

## Scope of this tutorial

This tutorial illustrates why expressive data models are crucial to produce high-fidelity models and trustworthy calculations for power distribution networks.

The [JSON data model](https://github.com/distribution-system-opt/bmopf-report) developed by the IEEE Task Force on Benchmarking Multiconductor OPF for Distribution Systems is used, together with the Julia package [BMOPFtools.jl](https://github.com/frederikgeth/BMOPFTools.jl): a library with functionalities to analyse benchmark datasets for distribution optimal power flow, with mathematical models that allow to run (optimal) power flow.

The following examples are presented hereafter:

* a simple introduction on how to get and read the data, and run an OPF
* an illustration of the value of descriptive impedance data in the phase space: capturing untransposed lines in your OPF
* OPF for a single-wire earth-return (SWER) network: how to solve 100% utility networks thanks to proper grounding and neutral definitions. This is complemented by a small sub-example that shows that  
* an example of using both grounding and descriptive impedance data and avoid Kron reduction when inappropriate

To prepare this notebook, extensive use was made of the tutorials and examples from the [BMOPFTools documentation](https://frederikgeth.github.io/BMOPFTools.jl/dev/) (selected, edited, and streamlined for the purposes of this workshop), which the interested user can refer to for additional material.


## First: Check that all required packages are installed correctly

In [1]:
using JuMP, Ipopt, Plots

In [2]:
import BMOPFTools as _BM

## Example 1: Getting and reading the data and running an OPF

Currently, there is no `official` repository containing the benchmark network data from the task force, but we will make use of one of CSIRO's four-wire low voltage data set hosted at the [BMOPFDraftData library](https://github.com/frederikgeth/BMOPFDraftData) (data as available on 13 August 2026).

### Reading the data in BMOPFTools

<img src="images/bmopf_flowchart.png" />

In [47]:
net_original = _BM.parse_bmopf("original_network/network_1_Feeder_1.json")

Dict{String, Any} with 10 entries:
  "voltage_source" => Dict{String, Any}("source"=>Dict{String, Any}("_pmd"=>Dic…
  "name"           => "network_1_Feeder_1"
  "line"           => Dict{String, Any}("line865"=>Dict{String, Any}("length"=>…
  "meta"           => Dict{String, Any}("\$schema"=>"https://raw.githubusercont…
  "shunt"          => Dict{String, Any}("grounding"=>Dict{String, Any}("B_1_1"=…
  "load"           => Dict{String, Any}("load24"=>Dict{String, Any}("p_nom"=>An…
  "generator"      => Dict{String, Any}("der_785"=>Dict{String, Any}("cost"=>An…
  "bus"            => Dict{String, Any}("1"=>Dict{String, Any}("terminal_names"…
  "linecode"       => Dict{String, Any}("lc7"=>Dict{String, Any}("X_series_2_3"…
  "_meta"          => Dict{String, Any}("parsed_at"=>"2026-08-16T23:34:14.697")

In [4]:
net_original["bus"]

Dict{String, Any} with 907 entries:
  "1"   => Dict{String, Any}("terminal_names"=>Any["1", "2", "3", "n"], "vpn_mi…
  "519" => Dict{String, Any}("terminal_names"=>Any["1", "2", "3", "n"], "vpn_mi…
  "788" => Dict{String, Any}("terminal_names"=>Any["1", "2", "3", "n"], "vpn_mi…
  "371" => Dict{String, Any}("terminal_names"=>Any["1", "2", "3", "n"], "vpn_mi…
  "774" => Dict{String, Any}("terminal_names"=>Any["1", "2", "3", "n"], "vpn_mi…
  "41"  => Dict{String, Any}("terminal_names"=>Any["1", "2", "3", "n"], "vpn_mi…
  "65"  => Dict{String, Any}("terminal_names"=>Any["1", "2", "3", "n"], "vpn_mi…
  "475" => Dict{String, Any}("terminal_names"=>Any["1", "2", "3", "n"], "vpn_mi…
  "705" => Dict{String, Any}("terminal_names"=>Any["1", "2", "3", "n"], "vpn_mi…
  "447" => Dict{String, Any}("terminal_names"=>Any["1", "2", "3", "n"], "vpn_mi…
  "593" => Dict{String, Any}("terminal_names"=>Any["1", "2", "3", "n"], "vpn_mi…
  "362" => Dict{String, Any}("terminal_names"=>Any["1", "2", "3", "n"], "

In [5]:
net_original["line"]

Dict{String, Any} with 906 entries:
  "line865" => Dict{String, Any}("length"=>0.090338, "bus_from"=>"859", "bus_to…
  "line617" => Dict{String, Any}("length"=>3.1088, "bus_from"=>"604", "bus_to"=…
  "line821" => Dict{String, Any}("length"=>0.15794, "bus_from"=>"814", "bus_to"…
  "line722" => Dict{String, Any}("length"=>0.083433, "bus_from"=>"717", "bus_to…
  "line120" => Dict{String, Any}("length"=>0.19481, "bus_from"=>"117", "bus_to"…
  "line330" => Dict{String, Any}("length"=>9.1332, "bus_from"=>"325", "bus_to"=…
  "line902" => Dict{String, Any}("length"=>0.18732, "bus_from"=>"902", "bus_to"…
  "line523" => Dict{String, Any}("length"=>0.066573, "bus_from"=>"515", "bus_to…
  "line632" => Dict{String, Any}("length"=>1.5703, "bus_from"=>"626", "bus_to"=…
  "line535" => Dict{String, Any}("length"=>0.12209, "bus_from"=>"528", "bus_to"…
  "line602" => Dict{String, Any}("length"=>4.6424, "bus_from"=>"594", "bus_to"=…
  "line415" => Dict{String, Any}("length"=>0.094921, "bus_from"=>"410", "

In [6]:
net_original["line"]["line865"]

Dict{String, Any} with 6 entries:
  "length"            => 0.090338
  "bus_from"          => "859"
  "bus_to"            => "866"
  "terminal_map_from" => Any["1", "2", "3", "n"]
  "terminal_map_to"   => Any["1", "2", "3", "n"]
  "linecode"          => "lc1"

In [7]:
net_original["linecode"]["lc1"]

Dict{String, Any} with 97 entries:
  "X_series_2_3" => 0.00070137
  "R_series_1_3" => 0.000196053
  "B_from_4_2"   => 0
  "R_series_1_2" => 0.000196222
  "R_series_2_4" => 0.000196053
  "B_to_4_1"     => 0
  "R_series_2_2" => 0.00134801
  "G_to_1_3"     => 0
  "G_from_2_2"   => 0
  "B_from_3_3"   => 0
  "G_to_2_1"     => 0
  "B_to_4_4"     => 0
  "B_from_2_2"   => 0
  "G_from_1_3"   => 0
  "B_from_3_2"   => 0
  "B_from_1_2"   => 0
  "B_from_3_1"   => 0
  "B_to_3_4"     => 0
  "X_series_3_4" => 0.000701402
  "X_series_2_1" => 0.000701397
  "B_from_1_3"   => 0
  "B_from_1_1"   => 0
  "B_to_2_3"     => 0
  "G_from_2_4"   => 0
  "R_series_2_1" => 0.000196222
  ⋮              => ⋮

### Bonus modelling notion: network (bus number) reduction

<img src="images/network-reduction.png" />

In [8]:
net_reduced = _BM.parse_bmopf("reduced_network/network_1_Feeder_1.json")

Dict{String, Any} with 11 entries:
  "bus"                 => Dict{String, Any}("101"=>Dict{String, Any}("terminal…
  "name"                => "network_1_Feeder_1"
  "generator"           => Dict{String, Any}("der_785"=>Dict{String, Any}("cost…
  "voltage_source"      => Dict{String, Any}("source"=>Dict{String, Any}("_pmd"…
  "line"                => Dict{String, Any}("line865"=>Dict{String, Any}("leng…
  "_simplification_log" => Any[Dict{String, Any}("message"=>"Removed dangling l…
  "_meta"               => Dict{String, Any}("parsed_at"=>"2026-08-16T11:32:09.…
  "meta"                => Dict{String, Any}("\$schema"=>"https://raw.githubuse…
  "shunt"               => Dict{String, Any}("grounding"=>Dict{String, Any}("B_…
  "load"                => Dict{String, Any}("load24"=>Dict{String, Any}("p_nom…
  "linecode"            => Dict{String, Any}("lc7"=>Dict{String, Any}("X_series…

## Solve an OPF problem

We minimize the total active-power generation cost **rate** 

$$\min \sum_{g \in \mathcal{G}} \sum_{k=1}^{|\mathcal{T}_g^\phi|}
  \frac{c^g_k}{1000} \cdot
  \bigl(\Delta v^r_k \, c^{r,g}_{g,k} + \Delta v^i_k \, c^{i,g}_{g,k}\bigr)$$

where $c^g_k$ (currency/kWh) is the **per-phase** energy price — the `cost` field is a vector with one entry per phase term, indexed by $k$ — and
$\Delta v_k$ is the phase-to-neutral (WYE) or line-to-line (DELTA) voltage at generator $g$'s $k$-th phase terminal. Division by 1000 converts the active-power expression from W to kW, so the snapshot objective has units currency/h. 

`**Note:** this is the OPF objective/model in BMOPF, of course variants exist (nonlinear cost, ...)`

In [48]:
optimizer = JuMP.optimizer_with_attributes(Ipopt.Optimizer, "print_level" => 0)
result = _BM.solve_opf(net_original; optimizer = optimizer, per_unit = true)

println("Termination : ", result["termination_status"])
println("Solve time : ", result["solve_time"], " s")
println("Cost rate   : ", round(result["objective"]; sigdigits = 6), " \$/h")


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

Termination : LOCALLY_SOLVED
Solve time : 5.911999940872192 s
Cost rate   : -0.289502 $/h


In [10]:
optimizer = JuMP.optimizer_with_attributes(Ipopt.Optimizer, "print_level" => 0)
result = _BM.solve_opf(net_reduced; optimizer = optimizer, per_unit = true)

println("Termination : ", result["termination_status"])
println("Solve time : ", result["solve_time"], " s")
println("Cost rate   : ", round(result["objective"]; sigdigits = 6), " \$/h")

Termination : LOCALLY_SOLVED
Solve time : 0.3840000629425049 s
Cost rate   : -0.289502 $/h


In [11]:
gen = result["generator"]["grid"]

Dict{String, Any} with 3 entries:
  "1" => Dict{String, Any}("crg"=>-435.867, "qg"=>255.548, "pg"=>-1.04686e5, "c…
  "2" => Dict{String, Any}("crg"=>222.456, "qg"=>-1254.08, "pg"=>-1.04686e5, "c…
  "3" => Dict{String, Any}("crg"=>221.915, "qg"=>1104.27, "pg"=>-1.04686e5, "ci…

In [12]:
gen_grid = sum(v["pg"] for v in values(gen))

-314056.821844666

In [13]:
src    = result["voltage_source"]["source"]   # your source id
p_grid = sum(v["ps"] for v in values(src))

367309.0191775337

## Example #2: capturing untransposed lines / cables

### Best-practice impedance matrices for four-wire networks

<img src="images/4-wire-cable.png" />

cable image from [Nexans](https://www.nexans.fr/en/business/buildings/building/Fast-installation/TWISTAL.html)

<img src="images/full-4-wire-impedances.png" />

##### Series resistance entries

In [20]:
for (k,v) in net_reduced["linecode"]["lc1"] if occursin("R", k) println(k, " value: ", round(v, digits=10)) end end

R_series_1_3 value: 0.0001960531
R_series_1_2 value: 0.0001962221
R_series_2_4 value: 0.0001960531
R_series_2_2 value: 0.0013480121
R_series_2_1 value: 0.0001962221
R_series_4_4 value: 0.0013480271
R_series_3_1 value: 0.0001960531
R_series_4_2 value: 0.0001960531
R_series_4_3 value: 0.0001962231
R_series_1_1 value: 0.0013480251
R_series_3_2 value: 0.0001962161
R_series_3_4 value: 0.0001962231
R_series_3_3 value: 0.0013480131
R_series_2_3 value: 0.0001962161
R_series_1_4 value: 0.0001962291
R_series_4_1 value: 0.0001962291


##### Series reactance entries

In [21]:
for (k,v) in net_reduced["linecode"]["lc1"] if occursin("X", k) println(k, " value: ", round(v, digits=10)) end end

X_series_2_3 value: 0.0007013702
X_series_3_4 value: 0.0007014022
X_series_2_1 value: 0.0007013972
X_series_4_3 value: 0.0007014022
X_series_3_3 value: 0.0007813722
X_series_4_4 value: 0.0007814382
X_series_2_2 value: 0.0007813812
X_series_1_2 value: 0.0007013972
X_series_2_4 value: 0.0006774342
X_series_1_3 value: 0.0006774312
X_series_1_4 value: 0.0007014262
X_series_3_2 value: 0.0007013702
X_series_4_2 value: 0.0006774342
X_series_1_1 value: 0.0007814252
X_series_4_1 value: 0.0007014262
X_series_3_1 value: 0.0006774312


##### Shunts

In [22]:
for (k,v) in net_reduced["linecode"]["lc1"] if occursin("G_to", k) println(k, " value: ", round(v, digits=10)) end end

G_to_1_3 value: 0.0
G_to_2_1 value: 0.0
G_to_3_3 value: 0.0
G_to_4_3 value: 0.0
G_to_2_3 value: 0.0
G_to_1_4 value: 0.0
G_to_3_1 value: 0.0
G_to_3_4 value: 0.0
G_to_2_2 value: 0.0
G_to_3_2 value: 0.0
G_to_4_2 value: 0.0
G_to_2_4 value: 0.0
G_to_1_2 value: 0.0
G_to_4_1 value: 0.0
G_to_1_1 value: 0.0
G_to_4_4 value: 0.0


In [23]:
for (k,v) in net_reduced["linecode"]["lc1"] if occursin("B_to", k) println(k, " value: ", round(v, digits=10)) end end

B_to_4_1 value: 0.0
B_to_4_4 value: 0.0
B_to_3_4 value: 0.0
B_to_2_3 value: 0.0
B_to_1_2 value: 0.0
B_to_2_1 value: 0.0
B_to_2_2 value: 0.0
B_to_1_1 value: 0.0
B_to_4_3 value: 0.0
B_to_3_2 value: 0.0
B_to_1_3 value: 0.0
B_to_2_4 value: 0.0
B_to_4_2 value: 0.0
B_to_1_4 value: 0.0
B_to_3_1 value: 0.0
B_to_3_3 value: 0.0


`BMOPFTools` has functionalities to build and inspect the matrices

In [25]:
X = _BM._pattern_keys_to_matrix(net_reduced["linecode"]["lc1"], "X_series_")

4×4 Matrix{Float64}:
 0.000781425  0.000701397  0.000677431  0.000701426
 0.000701397  0.000781381  0.00070137   0.000677434
 0.000677431  0.00070137   0.000781372  0.000701402
 0.000701426  0.000677434  0.000701402  0.000781438

<span style="background-color:yellow">**Q: is the cable in this linecode transposed?**</span>

<span style="background-color:yellow">**Q: can you inspect the series resistance matrix? What do you observe?** </span>

### Best practice vs impedance data in other datasets, datasheets, less-expressive in-house power flow tools, ...

Linecodes from the European LV test feeder From the [IEEE resource center](https://cmte.ieee.org/pes-testfeeders/resources/):

<img src="images/linecodes-ieee-lv-tf.png" />

<img src="images/flowchart-impedances.png" />

### Testing the impact of reduced impedance models with BMOPFTools

_This example is taken from the [impedance models tutorial of BMOPFTools](https://frederikgeth.github.io/BMOPFTools.jl/dev/tutorial_impedance_models/) (version 0.1.0, 14 August 2026)_

We work with a toy network with one single line: a 50 Hz, four-wire LV overhead feeder (a small AAC phase conductor on a 0.3 m cross-arm at 8 m, an offset neutral — an untransposed arrangement). Its transposed idealisation built by averaging. 

In [27]:
mm = 1e-3 # mm to meter conversion

0.001

**Step 1**: initialize the wire data and the line geometry (using coordinates of conductor centers)

In [28]:
toy = Dict{String,Any}(
    "wire_data" => Dict{String,Any}(
        "ph" => Dict{String,Any}("kind"=>"overhead","r_ac"=>0.9e-3,"radius"=>3.75mm,"gmr"=>2.9mm,"i_max"=>150.0),
        "nt" => Dict{String,Any}("kind"=>"overhead","r_ac"=>0.9e-3,"radius"=>3.75mm,"gmr"=>2.9mm,"i_max"=>150.0)),
    "line_geometry" => Dict{String,Any}("g" => Dict{String,Any}(
        "frequency"=>50.0, "earth_model"=>"modified_carson", "earth_resistivity"=>100.0,
        "conductors" => Any[
            Dict{String,Any}("wire_data"=>"ph","x"=>0.0,"y"=>8.0,"terminal"=>"a"),
            Dict{String,Any}("wire_data"=>"ph","x"=>0.3,"y"=>8.0,"terminal"=>"b"),
            Dict{String,Any}("wire_data"=>"ph","x"=>0.6,"y"=>8.0,"terminal"=>"c"),
            Dict{String,Any}("wire_data"=>"nt","x"=>0.3,"y"=>7.7,"terminal"=>"n")])))

Dict{String, Any} with 2 entries:
  "line_geometry" => Dict{String, Any}("g"=>Dict{String, Any}("earth_resistivit…
  "wire_data"     => Dict{String, Any}("nt"=>Dict{String, Any}("kind"=>"overhea…

**Step 2:** convert into a BMOPF dictionary; inspect the resulting matrices

In [30]:
_BM.compile_linecode(toy, "g")
lc_geo = toy["linecode"]["g"]

Dict{String, Any} with 68 entries:
  "X_series_2_3" => 0.000505236
  "R_series_1_3" => 4.9348e-5
  "B_from_4_2"   => -4.06046e-10
  "R_series_1_2" => 4.9348e-5
  "R_series_2_4" => 4.9348e-5
  "B_to_4_1"     => -3.37983e-10
  "R_series_2_2" => 0.000949348
  "B_from_3_3"   => 1.49137e-9
  "B_to_4_4"     => 1.53681e-9
  "B_from_2_2"   => 1.65617e-9
  "B_from_3_2"   => -4.39743e-10
  "B_from_1_2"   => -4.39743e-10
  "B_from_3_1"   => -2.30727e-10
  "B_to_3_4"     => -3.37983e-10
  "X_series_3_4" => 0.00048346
  "X_series_2_1" => 0.000505236
  "B_from_1_3"   => -2.30727e-10
  "B_from_1_1"   => 1.49137e-9
  "B_to_2_3"     => -4.39743e-10
  "R_series_2_1" => 4.9348e-5
  "R_series_4_4" => 0.000949348
  "R_series_3_1" => 4.9348e-5
  "X_series_4_3" => 0.00048346
  "X_series_3_3" => 0.000796717
  "B_to_1_2"     => -4.39743e-10
  ⋮              => ⋮

In [31]:
R = _BM._pattern_keys_to_matrix(lc_geo, "R_series_")

4×4 Matrix{Float64}:
 0.000949348  4.9348e-5    4.9348e-5    4.9348e-5
 4.9348e-5    0.000949348  4.9348e-5    4.9348e-5
 4.9348e-5    4.9348e-5    0.000949348  4.9348e-5
 4.9348e-5    4.9348e-5    4.9348e-5    0.000949348

In [32]:
X = _BM._pattern_keys_to_matrix(lc_geo, "X_series_")

4×4 Matrix{Float64}:
 0.000796717  0.000505236  0.000461684  0.00048346
 0.000505236  0.000796717  0.000505236  0.000505236
 0.000461684  0.000505236  0.000796717  0.00048346
 0.00048346   0.000505236  0.00048346   0.000796717

**Step 3:** apply transposition

*Transposition idealisation: average self, average phase-mutual, average phase-neutral*

In [33]:
function transposed_of(R, X)
    avg(M, S) = sum(M[i, j] for (i, j) in S) / length(S)
    d3, m3, pn = [(1,1),(2,2),(3,3)], [(1,2),(1,3),(2,3)], [(1,4),(2,4),(3,4)]
    sR, sX = avg(R, d3), avg(X, d3); mR, mX = avg(R, m3), avg(X, m3); pR, pX = avg(R, pn), avg(X, pn)
    lc = Dict{String,Any}()
    for i in 1:4, j in 1:4
        lc["R_series_$(i)_$(j)"] = i==j ? (i<=3 ? sR : R[4,4]) : (i<=3 && j<=3 ? mR : pR)
        lc["X_series_$(i)_$(j)"] = i==j ? (i<=3 ? sX : X[4,4]) : (i<=3 && j<=3 ? mX : pX)
    end
    lc
end

transposed_of (generic function with 1 method)

In [34]:
lc_bal = transposed_of(R, X)

Dict{String, Any} with 32 entries:
  "X_series_4_3" => 0.000490718
  "X_series_2_3" => 0.000490718
  "R_series_1_3" => 4.9348e-5
  "R_series_3_2" => 4.9348e-5
  "R_series_1_2" => 4.9348e-5
  "R_series_2_4" => 4.9348e-5
  "X_series_3_3" => 0.000796717
  "R_series_2_2" => 0.000949348
  "R_series_3_4" => 4.9348e-5
  "R_series_1_4" => 4.9348e-5
  "R_series_4_2" => 4.9348e-5
  "X_series_1_3" => 0.000490718
  "X_series_3_4" => 0.000490718
  "X_series_1_1" => 0.000796717
  "R_series_4_3" => 4.9348e-5
  "X_series_2_1" => 0.000490718
  "R_series_1_1" => 0.000949348
  "X_series_1_4" => 0.000490718
  "R_series_3_3" => 0.000949348
  "X_series_4_4" => 0.000796717
  "X_series_3_2" => 0.000490718
  "R_series_2_1" => 4.9348e-5
  "R_series_4_1" => 4.9348e-5
  "X_series_4_1" => 0.000490718
  "X_series_2_2" => 0.000796717
  ⋮              => ⋮

In [35]:
_BM._pattern_keys_to_matrix(lc_bal, "R_series_")

4×4 Matrix{Float64}:
 0.000949348  4.9348e-5    4.9348e-5    4.9348e-5
 4.9348e-5    0.000949348  4.9348e-5    4.9348e-5
 4.9348e-5    4.9348e-5    0.000949348  4.9348e-5
 4.9348e-5    4.9348e-5    4.9348e-5    0.000949348

In [36]:
_BM._pattern_keys_to_matrix(lc_bal, "X_series_")

4×4 Matrix{Float64}:
 0.000796717  0.000490718  0.000490718  0.000490718
 0.000490718  0.000796717  0.000490718  0.000490718
 0.000490718  0.000490718  0.000796717  0.000490718
 0.000490718  0.000490718  0.000490718  0.000796717

Compare with original X:

In [37]:
X

4×4 Matrix{Float64}:
 0.000796717  0.000505236  0.000461684  0.00048346
 0.000505236  0.000796717  0.000505236  0.000505236
 0.000461684  0.000505236  0.000796717  0.00048346
 0.00048346   0.000505236  0.00048346   0.000796717

**Step 4:** build a single-line feeder and run OPF on the transposed vs untransposed version

<img src="images/single-line-feeder.png"  width="400"/>

In [1]:
### support function to create the feeder

function feeder(lc; p_nom, len)
    Dict{String,Any}(
        "bus" => Dict{String,Any}(
            "src" => Dict{String,Any}("terminal_names"=>["a","b","c","n"], "perfectly_grounded_terminals"=>["n"]),
            "b1"  => Dict{String,Any}("terminal_names"=>["a","b","c","n"])),
        "voltage_source" => Dict{String,Any}("vs" => Dict{String,Any}("bus"=>"src","terminal_map"=>["a","b","c"],
            "v_magnitude"=>[230.0,230.0,230.0], "v_angle"=>[0.0,-2.0944,2.0944])),
        "linecode" => Dict{String,Any}("lc"=>lc),
        "line" => Dict{String,Any}("l1" => Dict{String,Any}("bus_from"=>"src","bus_to"=>"b1",
            "terminal_map_from"=>["a","b","c","n"], "terminal_map_to"=>["a","b","c","n"],
            "linecode"=>"lc", "length"=>len)),
        "load" => Dict{String,Any}("ld" => Dict{String,Any}("bus"=>"b1","terminal_map"=>["a","b","c","n"],
            "configuration"=>"WYE", "p_nom"=>p_nom, "q_nom"=>0.2 .* p_nom)))
end

feeder (generic function with 1 method)

$$VUF = \frac{|V^-|}{|V^+|}\times100 \, [\%]$$

In [2]:
### support function to calculate voltage unbalance factor (VUF) from phase-to-ground phasors in the OPF result
function vuf(r)
    b = r["bus"]["b1"]; a = exp(im*2pi/3)
    V = [b[t]["vm"]*exp(im*b[t]["va"]) for t in ("a","b","c")]
    Vp = (V[1] + a*V[2] + a^2*V[3])/3
    Vn = (V[1] + a^2*V[2] + a*V[3])/3
    abs(Vn)/abs(Vp)*100
end

vuf (generic function with 1 method)

Now let's solve a power flow, assigning one _balanced_ load to the feeders

In [40]:
rb = _BM.solve_pf(feeder(lc_bal; p_nom=[8e3,8e3,8e3], len=300.0))
rg = _BM.solve_pf(feeder(lc_geo; p_nom=[8e3,8e3,8e3], len=300.0)) 

Dict{String, Any} with 20 entries:
  "capacitor"          => Dict{String, Any}()
  "opt_profile"        => Dict{String, Any}("n_active"=>0, "min_active_multipli…
  "dc_branch"          => Dict{String, Any}()
  "objective"          => 0.0
  "bus"                => Dict{String, Any}("src"=>Dict{String, Any}("c"=>Dict{…
  "ground"             => Dict{String, Any}("src"=>Dict{String, Any}("n"=>Dict{…
  "losses"             => Dict{String, Any}("q_loss"=>380.354, "p_loss"=>1119.1…
  "switch"             => Dict{String, Any}()
  "generator"          => Dict{String, Any}()
  "dc_bus"             => Dict{String, Any}()
  "feasible"           => true
  "voltage_source"     => Dict{String, Any}("vs"=>Dict{String, Any}("c"=>Dict{S…
  "line"               => Dict{String, Any}("l1"=>Dict{String, Any}("c"=>Dict{S…
  "initialisation"     => Dict{String, Any}("src"=>Dict{String, Any}("c"=>Dict{…
  "termination_status" => "LOCALLY_SOLVED"
  "ibr"                => Dict{String, Any}()
  "transformer"   

Let's check the _VUF_ values in the two cases:

In [41]:
vuf(rb)

0.00028476115008783926

In [42]:
vuf(rg)

0.1470737924667688

<span style="background-color:yellow">**Now analyze `solve_time`and voltage magnitude (`vm`) of bus `b1`. Does it pay off to capture the untransposed nature of the line?** </span>

## Example 3: Single-wire earth return network

_This example is adapted from BMOPFTools' tutorials on [grounding](https://frederikgeth.github.io/BMOPFTools.jl/dev/tutorial_grounding/) and on [swer](https://frederikgeth.github.io/BMOPFTools.jl/dev/tutorial_swer/)_.

Expressive grounding information is crucial to develop utility tools solve 100% rather than 80% of their real-life networks.

`swer` is a single-wire earth return network with one perfectly-grounded source neutral, and five 1000 S (1 mΩ) electrode shunts along the feeder.

In [153]:
swer = _BM.from_dss(joinpath(pkgdir(_BM), "test", "data", "SWER", "Master.dss"))

┌ Warning: from_dss: 23 piece(s) of OpenDSS information could not be represented in BMOPF (full list on net["_meta"]["powerio_warnings"]):
│   linecode swer: `units` has no place in the BMOPF schema; dropped from the output
│   line l1: `units` has no place in the BMOPF schema; dropped from the output
│   line l2: `units` has no place in the BMOPF schema; dropped from the output
│   load ld_lv1: `kv` has no place in the BMOPF schema; dropped from the output
│   load ld_lv1: `phases` has no place in the BMOPF schema; dropped from the output
│   … and 18 more
└ @ BMOPFTools C:\Users\u0122389\.julia\packages\BMOPFTools\cRuBH\src\io\from_dss.jl:135


Dict{String, Any} with 11 entries:
  "terminal_conventions" => Dict{String, Any}("phase"=>["a", "b", "c"], "earth"…
  "bus"                  => Dict{String, Any}("mv"=>Dict{String, Any}("perfectl…
  "name"                 => "qld_swer"
  "voltage_source"       => Dict{String, Any}("source"=>Dict{String, Any}("v_ma…
  "line"                 => Dict{String, Any}("l2"=>Dict{String, Any}("length"=…
  "_meta"                => Dict{String, Any}("frequency_source"=>"powerio", "p…
  "meta"                 => Dict{String, Any}("\$schema"=>"https://raw.githubus…
  "shunt"                => Dict{String, Any}("grnd_swer2"=>Dict{String, Any}("…
  "transformer"          => Dict{String, Any}("single_phase"=>Dict{String, Any}…
  "load"                 => Dict{String, Any}("ld_lv2b"=>Dict{String, Any}("v_n…
  "linecode"             => Dict{String, Any}("swer"=>Dict{String, Any}("G_from…

<img src="images/swer_net_figure.png"  width="600"/>

In [39]:
swer["bus"]["lv_1"]

Dict{String, Any} with 2 entries:
  "terminal_names"   => ["a", "n"]
  "neutral_terminal" => "n"

In [69]:
println("convention: ", swer["terminal_conventions"])
println("SWER perfectly grounded : ",
        [(b, bd["perfectly_grounded_terminals"]) for (b, bd) in swer["bus"]
         if !isempty(get(bd, "perfectly_grounded_terminals", String[]))])
println("SWER electrodes (shunts): ",
        sort([(id, s["bus"], round(Float64(s["G_1_1"]); digits = 0)) for (id, s) in swer["shunt"]]))

convention: Dict{String, Any}("phase" => ["a", "b", "c"], "earth" => String[], "neutral" => ["n"])
SWER perfectly grounded : [("mv", ["n"])]
SWER electrodes (shunts): [("grnd_lv1", "lv_1", 1000.0), ("grnd_lv2", "lv_2", 1000.0), ("grnd_swer0", "swer_0", 1000.0), ("grnd_swer1", "swer_1", 1000.0), ("grnd_swer2", "swer_2", 1000.0)]


In [4]:
swer["transformer"]["single_phase"]["dx1"]

Dict{String, Any} with 13 entries:
  "x_series_to"       => 0
  "bus_from"          => "swer_1"
  "s_rating"          => 25000
  "terminal_map_to"   => ["a", "n"]
  "terminal_map_from" => ["a", "n"]
  "b_no_load"         => -0.00651042
  "r_series_from"     => 64.516
  "r_series_to"       => 0.02304
  "v_nom_to"          => 240
  "bus_to"            => "lv_1"
  "x_series_from"     => 258.064
  "v_nom_from"        => 12700
  "g_no_load"         => 0.00130208

We increase the length of this small test-network to 96 km and make increase conductor resistance

In [154]:
for (_, l) in swer["line"]; l["length"] *= 8.0; end       # 12 km → 96 km
swer["linecode"]["swer"]["R_series_1_1"] = 2.5 / 1000      # Ω/m

0.0025

And we add voltage limits for our OPF, e.g., $\pm$ 5% of the nominal V.

In [133]:
function set_limits!(n)
    for b in ("lv_1", "lv_2")
        nph = count(t -> t != "n", n["bus"][b]["terminal_names"])
        n["bus"][b]["vpn_min"] = fill(218.5, nph)
        n["bus"][b]["vpn_max"] = fill(241.5, nph)
    end
    for b in ("swer_0", "swer_1", "swer_2")
        n["bus"][b]["v_min"] = [11430.0]; n["bus"][b]["v_max"] = [13970.0]
    end
    n
end

set_limits! (generic function with 1 method)

#### Solve the SWER OPF

In [155]:
r = _BM.solve_opf(set_limits!(swer), optimizer = optimizer, per_unit = true)

Dict{String, Any} with 19 entries:
  "opt_profile"        => Dict{String, Any}("n_active"=>0, "min_active_multipli…
  "dc_branch"          => Dict{String, Any}()
  "objective"          => 0.0
  "bus"                => Dict{String, Any}("mv"=>Dict{String, Any}("c"=>Dict{S…
  "ground"             => Dict{String, Any}("mv"=>Dict{String, Any}("n"=>Dict{S…
  "losses"             => Dict{String, Any}("q_loss"=>1410.36, "p_loss"=>336.43…
  "switch"             => Dict{String, Any}()
  "generator"          => Dict{String, Any}()
  "dc_bus"             => Dict{String, Any}()
  "feasible"           => true
  "voltage_source"     => Dict{String, Any}("source"=>Dict{String, Any}("c"=>Di…
  "line"               => Dict{String, Any}("l2"=>Dict{String, Any}("ground"=>D…
  "initialisation"     => Dict{String, Any}("mv"=>Dict{String, Any}("c"=>Dict{S…
  "termination_status" => "LOCALLY_SOLVED"
  "ibr"                => Dict{String, Any}()
  "transformer"        => Dict{String, Any}("iso"=>Dict{String, 

In [156]:
# Phase-to-neutral voltage magnitude at bus `lv_1`, phase `a` from a result.
vpn_lv1 = (b = r["bus"]["lv_1"];
    abs((b["a"]["vr"] + im*b["a"]["vi"]) -
        (haskey(b, "n") ? b["n"]["vr"] + im*b["n"]["vi"] : 0.0 + 0im)))

237.8112569740655

#### DO: Solve a power flow after adding a 15 kVA PV generator to bus `lv_1`

In [157]:
### auxiliary function to add PV to a network 'n'

function add_pv(n, kw)
    n_pv = deepcopy(n)
    n_pv["ibr"] = get(n_pv, "ibr", Dict{String,Any}())
    n_pv["ibr"]["pv"] = Dict{String,Any}(
        "bus" => "lv_1", "terminal_map" => ["a","n"], "topology" => "SINGLE_PHASE",
        "prime_mover" => "PV",
        "s_max"   => [kw*1100.0],   # VA rating: 10 % headroom over P, typical sizing
        "p_min"   => [kw*1000.0],   # p_min = p_max pins P at full output —
        "p_max"   => [kw*1000.0],   #   a fixed injection, not a decision variable
        "q_min"   => [0.0],
        "q_max"   => [0.0],         # q pinned to 0: unity power factor
        "p_avail" => [kw*1000.0])   # irradiance-limited available power (= P here)
    return n_pv
end

add_pv (generic function with 1 method)

In [170]:
n_pv = add_pv(swer, 15.)

Dict{String, Any} with 12 entries:
  "terminal_conventions" => Dict{String, Any}("phase"=>["a", "b", "c"], "earth"…
  "bus"                  => Dict{String, Any}("mv"=>Dict{String, Any}("perfectl…
  "name"                 => "qld_swer"
  "voltage_source"       => Dict{String, Any}("source"=>Dict{String, Any}("v_ma…
  "line"                 => Dict{String, Any}("l2"=>Dict{String, Any}("length"=…
  "_meta"                => Dict{String, Any}("frequency_source"=>"powerio", "p…
  "ibr"                  => Dict{String, Any}("pv"=>Dict{String, Any}("p_avail"…
  "meta"                 => Dict{String, Any}("\$schema"=>"https://raw.githubus…
  "shunt"                => Dict{String, Any}("grnd_swer0"=>Dict{String, Any}("…
  "transformer"          => Dict{String, Any}("single_phase"=>Dict{String, Any}…
  "load"                 => Dict{String, Any}("ld_lv2b"=>Dict{String, Any}("v_n…
  "linecode"             => Dict{String, Any}("swer"=>Dict{String, Any}("G_from…

In [159]:
r = _BM.solve_pf(n_pv, optimizer = optimizer, per_unit = true)

Dict{String, Any} with 20 entries:
  "capacitor"          => Dict{String, Any}()
  "opt_profile"        => Dict{String, Any}("n_active"=>4, "min_active_multipli…
  "dc_branch"          => Dict{String, Any}()
  "objective"          => 0.0
  "bus"                => Dict{String, Any}("mv"=>Dict{String, Any}("c"=>Dict{S…
  "ground"             => Dict{String, Any}("mv"=>Dict{String, Any}("n"=>Dict{S…
  "losses"             => Dict{String, Any}("q_loss"=>1700.26, "p_loss"=>512.43…
  "switch"             => Dict{String, Any}()
  "generator"          => Dict{String, Any}()
  "dc_bus"             => Dict{String, Any}()
  "feasible"           => true
  "voltage_source"     => Dict{String, Any}("source"=>Dict{String, Any}("c"=>Di…
  "line"               => Dict{String, Any}("l2"=>Dict{String, Any}("ground"=>D…
  "initialisation"     => Dict{String, Any}("mv"=>Dict{String, Any}("c"=>Dict{S…
  "termination_status" => "LOCALLY_SOLVED"
  "ibr"                => Dict{String, Any}("pv"=>Dict{String, A

In [160]:
# Phase-to-neutral voltage magnitude at bus `lv_1`, phase `a` from a result.
vpn_lv1 = (b = r["bus"]["lv_1"];
    abs((b["a"]["vr"] + im*b["a"]["vi"]) -
        (haskey(b, "n") ? b["n"]["vr"] + im*b["n"]["vi"] : 0.0 + 0im)))

243.5803462691042

#### DO: add a 60 kVAr reactor to `swer_2` to see if that helps

In [161]:
add_reactor!(n, kvar) = (n["shunt"] = get(n, "shunt", Dict{String,Any}());
    n["shunt"]["reac"] = Dict{String,Any}("bus"=>"swer_2", "terminal_map"=>["a"],
        "B_1_1" => -kvar*1000.0 / 12700.0^2); n)   # fixed inductive susceptance (S)

add_reactor! (generic function with 1 method)

In [171]:
r_r = _BM.solve_pf(add_reactor!(n_pv, 60.); optimizer=optimizer, per_unit=true)

Dict{String, Any} with 20 entries:
  "capacitor"          => Dict{String, Any}()
  "opt_profile"        => Dict{String, Any}("n_active"=>4, "min_active_multipli…
  "dc_branch"          => Dict{String, Any}()
  "objective"          => 0.0
  "bus"                => Dict{String, Any}("mv"=>Dict{String, Any}("c"=>Dict{S…
  "ground"             => Dict{String, Any}("mv"=>Dict{String, Any}("n"=>Dict{S…
  "losses"             => Dict{String, Any}("q_loss"=>4552.75, "p_loss"=>5861.5…
  "switch"             => Dict{String, Any}()
  "generator"          => Dict{String, Any}()
  "dc_bus"             => Dict{String, Any}()
  "feasible"           => true
  "voltage_source"     => Dict{String, Any}("source"=>Dict{String, Any}("c"=>Di…
  "line"               => Dict{String, Any}("l2"=>Dict{String, Any}("ground"=>D…
  "initialisation"     => Dict{String, Any}("mv"=>Dict{String, Any}("c"=>Dict{S…
  "termination_status" => "LOCALLY_SOLVED"
  "ibr"                => Dict{String, Any}("pv"=>Dict{String, A

In [167]:
# check the voltage magnitude again
vpn_lv1 = (b = r_r["bus"]["lv_1"];
    abs((b["a"]["vr"] + im*b["a"]["vi"]) -
        (haskey(b, "n") ? b["n"]["vr"] + im*b["n"]["vi"] : 0.0 + 0im)))

234.353765447906

#### But do grounding models, in general, make a difference?

In [173]:
# define a simple linecode
lc4 = Dict{String,Any}()
for i in 1:4, j in 1:4
    lc4["R_series_$(i)_$(j)"] = (i == j ? 0.5 : 0.02) / 1000   # Ω/m
    lc4["X_series_$(i)_$(j)"] = (i == j ? 0.2 : 0.05) / 1000
end

In [176]:
# create a small feeder with an :electrode vs :perfect grounding

function feeder(ground)
    net = Dict{String,Any}(
        "bus" => Dict{String,Any}(
            "src" => Dict{String,Any}("terminal_names" => ["a","b","c","n"],
                                      "perfectly_grounded_terminals" => ["n"]),
            "lb"  => Dict{String,Any}("terminal_names" => ["a","b","c","n"],
                     "perfectly_grounded_terminals" =>
                         ground == :perfect ? ["n"] : String[])),
        "voltage_source" => Dict{String,Any}("source" => Dict{String,Any}(
            "bus" => "src", "terminal_map" => ["a","b","c","n"],
            "v_magnitude" => [230.0, 230.0, 230.0, 0.0],
            "v_angle" => [0.0, -2π/3, 2π/3, 0.0])),
        "linecode" => Dict{String,Any}("lc4" => deepcopy(lc4)),
        "line" => Dict{String,Any}("l1" => Dict{String,Any}(
            "bus_from" => "src", "bus_to" => "lb", "linecode" => "lc4",
            "length" => 500.0,
            "terminal_map_from" => ["a","b","c","n"],
            "terminal_map_to"   => ["a","b","c","n"])),
        "load" => Dict{String,Any}("ld" => Dict{String,Any}(
            "bus" => "lb", "terminal_map" => ["a","n"],
            "configuration" => "SINGLE_PHASE",
            "p_nom" => [3000.0], "q_nom" => [1000.0])))
    ground == :electrode && (net["shunt"] = Dict{String,Any}(
        "rod" => Dict{String,Any}("bus" => "lb", "terminal_map" => ["n"],
                                  "G_1_1" => 0.1)))        # a 10 Ω rod
    net
end

feeder (generic function with 1 method)

#### DO: solve power flow with either mode and compare the voltage magnitudes (and angles!) at bus `lb`, and the current through the neutral of line `l1`

In [178]:
# solution
for g in (:electrode, :perfect)
    r  = _BM.solve_pf(feeder(g); optimizer = optimizer)
    lb = r["bus"]["lb"]
    vn = abs(lb["n"]["vr"] + im*lb["n"]["vi"])
    va = abs((lb["a"]["vr"] + im*lb["a"]["vi"]) - (lb["n"]["vr"] + im*lb["n"]["vi"]))
    println(rpad(g, 12), lpad(round(vn; digits = 2), 6), " V",
            lpad(round(va; digits = 2), 10), " V",
            lpad(round(r["line"]["l1"]["n"]["cm_fr"]; digits = 2), 10), " A")
end

electrode     3.48 V    222.95 V     13.85 A
perfect        0.0 V    226.24 V       1.4 A


## Example 4: Kron reduction of the neutral

## TODO Marta: hide spoilers? https://discourse.jupyter.org/t/hiding-spoilers-answers-cells/4252/2